# RegionLens API — быстрый старт

Практические примеры обращения к API RegionLens из Python. Показано, как получить рейтинг
регионов, разложить изменение индекса по доменам, собрать индекс со своими весами и
смоделировать сценарий «что если». В конце — про аутентификацию по токену и лимиты частоты.

**Требуется:** запущенный сервер RegionLens и библиотека `requests` (`pip install requests`).

Канонический префикс API — `/api/v1/`. Полная интерактивная документация — по адресу
`/api/v1/docs/` вашего сервера.


## Настройка

Укажите адрес сервера. Токен (кабинет → «API-доступ» → «Выпустить токен») не обязателен для
чтения открытых данных, но идентифицирует запросы как ваши и повышает лимит частоты
(120 → 600 запросов в минуту). Чтобы работать с токеном — впишите его в `TOKEN`.


In [ ]:
import requests

BASE = "http://127.0.0.1:8000/api/v1"   # адрес вашего сервера
TOKEN = None  # например: "xxxxxxxx..." из кабинета → «API-доступ»

def api(path, **params):
    """GET-запрос к API. Токен, если задан, уходит в заголовок Authorization."""
    headers = {"Authorization": f"Token {TOKEN}"} if TOKEN else {}
    resp = requests.get(f"{BASE}{path}", params=params, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json()


## 1. Список регионов

Каждый регион описан кодом ОКАТО, названием и федеральным округом. Составим заодно словарь
`ОКАТО → название` — он пригодится дальше (в рейтинге индекса имена не дублируются).


In [ ]:
regions = api("/regions/")
print(len(regions), "регионов")

name_by_okato = {r["okato"]: r["region_name"] for r in regions}
regions[:3]


## 2. Рейтинг индекса развития

Композитный индекс за год по выбранной схеме весов (`equal` — равные веса, `pca`, `expert`).
Строки уже отсортированы по месту; `total_score` — балл 0–100, остальные поля — доменные баллы.


In [ ]:
ranking = api("/index/", year=2024, scheme="equal")
for row in ranking[:10]:
    name = name_by_okato.get(row["okato"], row["okato"])
    print(f'{row["rank"]:>3}. {name:<40} {row["total_score"]:.1f}')


## 3. Индекс со своими весами

Пересобрать рейтинг под собственные приоритеты: передайте вес каждого домена параметром
`w_<домен>`. Ответ содержит новое место (`rank`), место при равных весах (`base_rank`) и
сдвиг (`delta`). Ниже усилим экономику вдвое и посмотрим, кто поднялся сильнее всего.


In [ ]:
custom = api(
    "/index/custom/", year=2024,
    w_economy=2, w_income=1, w_demography=1, w_labor=1, w_infrastructure=1, w_health_edu=1,
)
movers = sorted(custom, key=lambda r: r["delta"], reverse=True)[:5]
for r in movers:
    print(f'{r["region_name"]:<40} место {r["rank"]:>3} (было {r["base_rank"]:>3}, сдвиг {r["delta"]:+d})')


## 4. Декомпозиция изменения индекса

Разложение годового изменения индекса региона по доменам: какой домен внёс вклад в рост или
спад. Возьмём первый регион из списка.


In [ ]:
okato = regions[0]["okato"]
decomposition = api("/decomposition/", okato=okato, year=2024)
print(name_by_okato[okato])
decomposition


## 5. Сценарий «что если»

Смоделируем: что будет с местом региона, если «подтянуть» его инфраструктуру до 90-го
перцентиля среди регионов. Ответ содержит базовое и сценарное место, а также анализ
чувствительности — какой домен даёт наибольший подъём.


In [ ]:
scenario = api("/index/scenario/", year=2024, okato=okato, p_infrastructure=90)
print(f'место: {scenario["baseline_rank"]} → {scenario["scenario_rank"]} '
      f'(из {scenario["of"]}, сдвиг {scenario["delta"]:+d})')
print("наибольший подъём дают домены:")
for s in scenario["sensitivity"][:3]:
    print(f'  {s["domain"]:<16} +{s["gain"]} мест')


## 6. Аутентификация по токену и лимиты частоты

Данные открыты — читать API можно и анонимно. Токен нужен для программной работы «как вы»:
он идентифицирует запросы и повышает лимит частоты (120/мин у анонима → 600/мин у
владельца токена). Токен передаётся заголовком `Authorization: Token <ключ>`.

Проверим, что метод подключён: заведомо неверный токен отклоняется с кодом **401**.


In [ ]:
bad = requests.get(f"{BASE}/regions/", headers={"Authorization": "Token неверный-ключ"})
print("неверный токен →", bad.status_code, "(ожидается 401)")


---

Полный перечень эндпойнтов, параметров и схем ответов — в интерактивной документации
`/api/v1/docs/` (Swagger UI) или в машиночитаемой схеме `/api/v1/schema/` (OpenAPI 3).
